+# Day 27 Mini Project — PDF Semantic Search

**Objective:** Build a beginner-friendly semantic search system.

**Pipeline:** PDF → Text Extraction → Chunking → Embeddings → Query Embedding → Cosine Similarity → Top-K Results

Upload your own PDF in Colab.

In [1]:
!pip -q install pypdf sentence-transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 7.3 MB/s eta 0:00:00


## 1. Upload a PDF

In [2]:
from google.colab import files

uploaded = files.upload()
pdf_path = next(iter(uploaded))
print("Uploaded:", pdf_path)

Saving AI Day 27.pdf to AI Day 27 (1).pdf
Uploaded: AI Day 27 (1).pdf


## 2. Extract Text

In [3]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)
pages = []

for page in reader.pages:
    text = page.extract_text()
    if text:
        pages.append(text)

full_text = "\n".join(pages)

print("Pages:", len(reader.pages))
print("Characters:", len(full_text))
print("\nPreview:\n", full_text[:2000])

Pages: 17
Characters: 10914

Preview:
 Day
 
27
 
 
Artiﬁcial
 
Intelligence
 
 
Embeddings
 
&
 
Vector
 
Databases
 
1.
 
What
 
is
 
an
 
Embedding?
 
An
 
embedding
 
is
 
a
 
numerical
 
representation
 
of
 
data
 
such
 
as
 
text,
 
images,
 
or
 
audio
 
in
 
the
 
form
 
of
 
a
 
vector.
 
Simple
 
Deﬁnition
 
Embedding
 
=
 
Converting
 
information
 
into
 
numbers
 
that
 
capture
 
its
 
meaning.
 
Example:
 
"Machine
 
Learning"
 
        
↓
 
Embedding
 
Model
 
        
↓
 
[0.21,
 
-0.54,
 
0.73,
 
0.18,
 
...]
 
 
2.
 
Why
 
Do
 
We
 
Need
 
Embeddings?
 
Computers
 
don't
 
directly
 
understand
 
the
 
meaning
 
of
 
sentences
 
like
 
humans
 
do.
 
Consider:
 
"I
 
love
 
programming"
 
and
 
"Programming
 
is
 
something
 
I
 
enjoy"
 
The
 
words
 
are
 
different,
 
but
 
the
 
meaning
 
is
 
similar
.
 
Embeddings
 
can
 
represent
 
their
 
semantic
 
similarity.
 
 
3.
 
Vector
 
A
 
vector
 
is
 
a
 
list
 
of
 
numbers.
 
Example:
 
[0.21,
 
0.45,
 
-0.17

## 3. Create Text Chunks

In [4]:
def create_chunks(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

chunks = create_chunks(full_text)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i} ---")
    print(chunk[:500])

Number of chunks: 28

--- Chunk 0 ---
Day
 
27
 
 
Artiﬁcial
 
Intelligence
 
 
Embeddings
 
&
 
Vector
 
Databases
 
1.
 
What
 
is
 
an
 
Embedding?
 
An
 
embedding
 
is
 
a
 
numerical
 
representation
 
of
 
data
 
such
 
as
 
text,
 
images,
 
or
 
audio
 
in
 
the
 
form
 
of
 
a
 
vector.
 
Simple
 
Deﬁnition
 
Embedding
 
=
 
Converting
 
information
 
into
 
numbers
 
that
 
capture
 
its
 
meaning.
 
Example:
 
"Machine
 
Learning"
 
        
↓
 
Embedding
 
Model
 
        
↓
 
[0.21,
 
-0.54,
 
0.73,
 
0.18,
 
...]

--- Chunk 1 ---
earning"
 
        
↓
 
Embedding
 
Model
 
        
↓
 
[0.21,
 
-0.54,
 
0.73,
 
0.18,
 
...]
 
 
2.
 
Why
 
Do
 
We
 
Need
 
Embeddings?
 
Computers
 
don't
 
directly
 
understand
 
the
 
meaning
 
of
 
sentences
 
like
 
humans
 
do.
 
Consider:
 
"I
 
love
 
programming"
 
and
 
"Programming
 
is
 
something
 
I
 
enjoy"
 
The
 
words
 
are
 
different,
 
but
 
the
 
meaning
 
is
 
similar
.
 
Embeddings
 
can
 
represent
 
their
 
semanti

## 4. Generate Embeddings

In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (28, 384)


## 5. Semantic Search

In [6]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def semantic_search(query, top_k=3):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores = cosine_similarity(query_embedding, embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]

    return [
        {
            "chunk_id": int(i),
            "score": float(scores[i]),
            "text": chunks[i]
        }
        for i in top_indices
    ]

## 6. Ask a Question

In [7]:
query = input("Enter your question: ")
results = semantic_search(query, top_k=3)

for result in results:
    print("\n" + "=" * 80)
    print("Chunk ID:", result["chunk_id"])
    print("Similarity Score:", round(result["score"], 4))
    print("-" * 80)
    print(result["text"])

Enter your question: what is a vector database

Chunk ID: 7
Similarity Score: 0.8426
--------------------------------------------------------------------------------
ocuments
 
about:
 
cars
 
vehicles
 
automobiles
 
even
 
if
 
the
 
exact
 
word
 
doesn't
 
match.
 
 
8.
 
What
 
is
 
a
 
Vector
 
Database?
 
A
 
Vector
 
Database
 
is
 
a
 
database
 
designed
 
to
 
store
 
and
 
efﬁciently
 
search
 
vector
 
embeddings.
 
Simple
 
Deﬁnition
 
Vector
 
Database
 
=
 
Database
 
optimized
 
for
 
storing
 
and
 
searching
 
vectors.
 
Examples:
 
●
 
Pinecone
 
●
 
Chroma
 
●
 
Weaviate
 
●
 
Milvus
 
●
 
Qdrant
 
●
 
pgvector
 
with
 
PostgreSQL/Supaba

Chunk ID: 21
Similarity Score: 0.6364
--------------------------------------------------------------------------------
mparison
 
Less
 
semantic
 
understanding
 
Better
 
semantic
 
retrieval
 
SQL/text
 
search
 
Vector
 
similarity
 
search
 
 
21.
 
Popular
 
Vector
 
Databases
 
Database
 
Type
 
Pinecone
 
Managed
 
vector


## 7. Try Multiple Questions

In [8]:
questions = [
    "What is the main topic of this document?",
    "What are the important concepts discussed?",
    "What information is provided about the main subject?"
]

for query in questions:
    print("\n" + "#" * 80)
    print("QUERY:", query)

    for result in semantic_search(query, top_k=2):
        print(f"\nScore: {result['score']:.4f}")
        print(result["text"][:700])


################################################################################
QUERY: What is the main topic of this document?

Score: 0.2924
earch
 
●
 
Document
 
search
 
●
 
Recommendation
 
systems
 
●
 
Question
 
answering
 
●
 
Code
 
search
 
●
 
E-commerce
 
search
 
●
 
Knowledge
 
bases
 
●
 
Customer-support
 
systems
 
 
⚡
 
Quick
 
Revision
 
Remember
 
this
 
pipeline:
 
DOCUMENT
 
   
↓
 
CHUNK
 
   
↓
 
EMBED
 
   
↓
 
VECTOR
 
   
↓
 
VECTOR
 
DATABASE
 
   
↓
 
SIMILARITY
 
SEARCH
 
   
↓
 
RELEVANT
 
CONTEXT
 
   
↓
 
LLM
 
   
↓
 
ANSWER
 
⭐
 
Most
 
Important
 
Points
 
1.
 
Embedding
 
=
 
numerical
 
represent

Score: 0.2463
ocuments
 
into
 
smaller
 
pieces.
 
7.
 
Top-K
 
retrieves
 
the
 
most
 
relevant
 
chunks.
 
8.
 
Embeddings
 
+
 
Vector
 
DBs
 
are
 
fundamental
 
components
 
of
 
RAG.
 
 
 
 
 
 
 
 
🎯
 
Day
 
27
 
Mini
 
Project
 
PDF
 
Semantic
 
Search
 
Build
 
a
 
simple
 
application:
 
Upload
 
PDF
 
    
↓
 
Extract
 
Text
 
    
↓
 
Ch

## 8. Understand the Pipeline

```text
User Question
      ↓
Query Embedding
      ↓
Compare with Document Embeddings
      ↓
Cosine Similarity
      ↓
Top-K Relevant Chunks
```

This is the **retrieval** part of a RAG system.

## 9. Practice

Try:

1. `chunk_size=300`
2. `overlap=50`
3. `top_k=5`
4. Upload a different PDF.
5. Ask the same question using different wording.

Example:

> What is machine learning?

Try:

> Explain the ML concept discussed in the document.

## 10. Key Takeaways

- **Embedding:** converts text into vectors.
- **Chunking:** splits documents into smaller pieces.
- **Cosine similarity:** compares vectors.
- **Semantic search:** searches by meaning.
- **Top-K:** retrieves the most relevant chunks.
- **Vector databases:** store embeddings for scalable retrieval.

**Next:** Connect retrieval to an LLM and build a complete RAG application.